# ScreamingFace ↔ URL4 engine

Combine three model routes into one URL4-backed fusion, run it on deterministic benchmark
questions, and measure whether the majority beats the best individual member.

This is simulated but uses the **real URL4 HTTP engine**. Before running the notebook, start its
deterministic routes from `packages/screamingface`:

```bash
./scripts/dev-url4.sh
```

To fetch GPQA Diamond instead of the bundled fixture, first accept its gated dataset terms and be
logged in to Hugging Face, then select the live dataset with `sf.config(mode="live")`. Your URL4
engine must also expose production-backed model routes.

The saved run uses deterministic routes, so its result is reproducible and makes no
provider-quality claim.

## 1 · Import the SDK

In [1]:
import screamingface as sf

# Optional: point the SDK at a hosted engine instead of the localhost default.
# sf.config("https://url4.example")

## 2 · Compose a fusion — Python or YAML

These are two representations of the same fusion. Use Python while exploring; use YAML when you
want a small configuration file to review or share.

### Option A · Python

In [2]:
fusion = sf.Fusion(
    "frontier-trio",
    models=[
        "codex/gpt-5.5",
        "gemini-cli/gemini-2.5-pro",
        "anthropic/claude-sonnet-4-6",
    ],
    reducer=sf.MajorityVote(tie_breaker="codex/gpt-5.5"),
)
fusion

Role,Model
Tie breaker,codex/gpt-5.5
Member,gemini-cli/gemini-2.5-pro
Member,anthropic/claude-sonnet-4-6


### Option B · YAML

The equivalent [`fusion.yaml`](fusion.yaml) is:

```yaml
name: frontier-trio
models:
  - codex/gpt-5.5
  - gemini-cli/gemini-2.5-pro
  - anthropic/claude-sonnet-4-6
reduce: majority_vote
tie_breaker: codex/gpt-5.5
```

In [3]:
fusion_from_yaml = sf.Fusion.from_yaml("fusion.yaml")
fusion_from_yaml.url4 == fusion.url4

True

## 3 · Inspect the shareable recipe

The recipe contains model routes and an unresolved `$question`. Constructing or displaying it
sends nothing. Evaluation binds each concrete question later.

In [4]:
fusion.url4

"(panel_1=/codex/gpt-5.5($question)!'Answer the multiple-choice question', panel_2=/gemini/2.5($question)!'Answer the multiple-choice question', panel_3=/claude/sonnet-4.6($question)!'Answer the multiple-choice question', {schema: 'screamingface.panel-result.v1', panel_1_model: 'codex/gpt-5.5', panel_1_answer: '$panel_1', panel_2_model: 'gemini-cli/gemini-2.5-pro', panel_2_answer: '$panel_2', panel_3_model: 'anthropic/claude-sonnet-4-6', panel_3_answer: '$panel_3'})"

## 4 · Run through the URL4 engine

For each question, ScreamingFace sends one complete expression to
`http://127.0.0.1:4404/v1`. The engine executes all three model routes and returns their labeled
answers. ScreamingFace never calls those routes, AI Gateway, or providers directly.

In [5]:
# For each question: GET http://127.0.0.1:4404/v1?q=<URL-encoded fusion expression>
# Decoded q expression:
# (question='<resolved GPQA prompt>',
#  panel_1=/codex/gpt-5.5($question)!'Answer the multiple-choice question',
#  panel_2=/gemini/2.5($question)!'Answer the multiple-choice question',
#  panel_3=/claude/sonnet-4.6($question)!'Answer the multiple-choice question',
#  {schema: 'screamingface.panel-result.v1',
#   panel_1_model: 'codex/gpt-5.5', panel_1_answer: '$panel_1',
#   panel_2_model: 'gemini-cli/gemini-2.5-pro', panel_2_answer: '$panel_2',
#   panel_3_model: 'anthropic/claude-sonnet-4-6', panel_3_answer: '$panel_3'})
#
# Compiled URL4 request node (↖ shared = the same binding, not another request):
# GatherNode
# ├─ question: BindingNode → TextNode '<resolved GPQA prompt>'
# ├─ panel_1: BindingNode → RelUrlNode /codex/gpt-5.5
# │  ├─ context → question ↖ shared
# │  └─ intent → TextNode 'Answer the multiple-choice question'
# ├─ panel_2: BindingNode → RelUrlNode /gemini/2.5
# │  ├─ context → question ↖ shared
# │  └─ intent → TextNode 'Answer the multiple-choice question'
# ├─ panel_3: BindingNode → RelUrlNode /claude/sonnet-4.6
# │  ├─ context → question ↖ shared
# │  └─ intent → TextNode 'Answer the multiple-choice question'
# └─ response: StructNode
#    ├─ schema → screamingface.panel-result.v1
#    ├─ panel_1_model → codex/gpt-5.5
#    ├─ panel_1_answer → panel_1 ↖ shared
#    ├─ panel_2_model → gemini-cli/gemini-2.5-pro
#    ├─ panel_2_answer → panel_2 ↖ shared
#    ├─ panel_3_model → anthropic/claude-sonnet-4-6
#    └─ panel_3_answer → panel_3 ↖ shared
run = fusion.evaluate("gpqa", first=20, seed=0)
run

Run(benchmark='GPQA-shaped synthetic science fixture', dataset_source='synthetic-gpqa-shaped', mode='mock', models=('codex/gpt-5.5', 'gemini-cli/gemini-2.5-pro', 'anthropic/claude-sonnet-4-6'), url="(panel_1=/codex/gpt-5.5($question)!'Answer the multiple-choice question', panel_2=/gemini/2.5($question)!'Answer the multiple-choice question', panel_3=/claude/sonnet-4.6($question)!'Answer the multiple-choice question', {schema: 'screamingface.panel-result.v1', panel_1_model: 'codex/gpt-5.5', panel_1_answer: '$panel_1', panel_2_model: 'gemini-cli/gemini-2.5-pro', panel_2_answer: '$panel_2', panel_3_model: 'anthropic/claude-sonnet-4-6', panel_3_answer: '$panel_3'})", sample_size=20, seed=0, score=100.0, baseline=80.0, gain=20.0, cost_usd=0.0, fusion_name='frontier-trio', reducer='majority_vote', tie_breaker='codex/gpt-5.5', incomplete=0, profiles=(), pricing_source='engine response does not yet report usage', pricing_as_of='n/a', prompt_tokens=0, completion_tokens=0, total_tokens=0, model_results=(ModelResult(model='codex/gpt-5.5', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0), ModelResult(model='gemini-cli/gemini-2.5-pro', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0), ModelResult(model='anthropic/claude-sonnet-4-6', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0)), failures=())

## 5 · Compare

In [6]:
{
    "sample_size": run.sample_size,
    "score": run.score,
    "baseline": run.baseline,
    "gain": run.gain,
}

{'sample_size': 20, 'score': 100.0, 'baseline': 80.0, 'gain': 20.0}

> `gain = fusion score − best member score` on the same answers. Positive gain
means the combination corrected mistakes made by every individual panel member.

The URL4 engine owns model execution. ScreamingFace owns majority vote, answer-key scoring,
baseline, and gain. Real AI-Gateway-backed model routes can replace the deterministic commands
later without changing this SDK flow.